# SQL Agent: Natural Language to SQL
## A Hands-On Workshop

A **SQL agent** lets you ask questions in plain English and get answers from a database — no SQL required from the user. Under the hood it uses the **ReAct** loop: the LLM reasons about what to do, calls a tool (like `run_sql`), reads the result, and repeats until it has a final answer.

By the end of this workbook you will have:
- Built a local SQLite database and seeded it with sales data
- Written three SQL tools from scratch
- Created a ReAct agent with LangGraph's `create_react_agent`
- Queried the agent in natural language and watched it reason step-by-step

In [ ]:
# Install dependencies -- only runs on Google Colab.
# Local users: your .venv already has everything from requirements.txt
import sys

if "google.colab" in sys.modules:
    %pip install -q langchain-openai langgraph langchain-core python-dotenv

In [ ]:
import os

from dotenv import load_dotenv

# Local: reads OPENAI_API_KEY from .env
# Colab: paste your key when prompted
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    import getpass

    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

---

## How ReAct Works

**ReAct** (Reason + Act) is a pattern where the LLM alternates between thinking and doing:

```
User question
     |
     v
+-----------------------------+
| THOUGHT: what do I need?   |  <- LLM reasons
| ACTION:  call a tool       |  <- LLM picks a tool + args
| OBSERVATION: tool result   |  <- result fed back to LLM
+-----------------------------+
     |  (repeat until done)
     v
  Final answer
```

For a SQL agent the tools are: **list tables** -> **describe schema** -> **run a query**.
The LLM discovers the schema by itself and writes the SQL -- you never hardcode a query.

---

## Part 1 -- The Database

We will use **SQLite** -- it runs entirely in a file, no server needed.
The database has one table: `sales`, with 8 rows covering three products, four regions, and four months.

In [ ]:
import sqlite3

DB_PATH = "sales.db"


def seed_db():
    conn = sqlite3.connect(DB_PATH)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS sales (
            id      INTEGER PRIMARY KEY,
            product TEXT,
            region  TEXT,
            amount  REAL,
            date    TEXT
        );
        INSERT OR IGNORE INTO sales VALUES
            (1, 'Widget A', 'North',  1200.00, '2024-01-15'),
            (2, 'Widget B', 'South',   850.50, '2024-01-20'),
            (3, 'Widget A', 'East',   2100.00, '2024-02-05'),
            (4, 'Widget C', 'North',   430.00, '2024-02-18'),
            (5, 'Widget B', 'West',   1750.00, '2024-03-01'),
            (6, 'Widget A', 'South',   990.00, '2024-03-22'),
            (7, 'Widget C', 'East',    610.00, '2024-04-10'),
            (8, 'Widget B', 'North',  1340.00, '2024-04-28');
    """)
    conn.commit()
    conn.close()


seed_db()
print("Database seeded:", DB_PATH)

In [ ]:
# Inspect the raw data so we know exactly what we are working with
conn = sqlite3.connect(DB_PATH)
rows = conn.execute("SELECT * FROM sales").fetchall()
conn.close()

header = f"{'id':>3}  {'product':<10} {'region':<8} {'amount':>8}  {'date'}"
print(header)
print("-" * len(header))
for r in rows:
    print(f"{r[0]:>3}  {r[1]:<10} {r[2]:<8} {r[3]:>8.2f}  {r[4]}")

---

## Part 2 -- SQL Tools

The agent needs tools that let it explore and query the database.
We build three, from broad to specific:

| Tool | What it does | When the agent uses it |
|------|-------------|------------------------|
| `list_tables` | Returns all table names | First -- discover what exists |
| `describe_table` | Returns column names and types | Second -- understand the schema |
| `run_sql` | Executes a SELECT query | Third -- get the actual data |

Each tool is decorated with `@tool`. The **docstring is critical** -- the LLM reads it to decide when and how to call each tool.

In [ ]:
from langchain_core.tools import tool


@tool
def list_tables() -> str:
    """List all tables in the database."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    conn.close()
    return ", ".join(r[0] for r in rows) or "No tables found."


@tool
def describe_table(table: str) -> str:
    """Return the column schema of a table (column name and data type)."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(f"PRAGMA table_info({table})").fetchall()
    conn.close()
    return "\n".join(f"{r[1]} ({r[2]})" for r in rows)


# Quick sanity check before wiring tools to the agent
print("Tables :", list_tables.invoke({}))
print("\nSchema :")
print(describe_table.invoke({"table": "sales"}))

In [ ]:
@tool
def run_sql(query: str) -> str:
    """Execute a read-only SQL SELECT query against the database. Returns up to 20 rows."""
    # Safety guard: block any write operations
    forbidden = {"INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE"}
    if any(kw in query.upper() for kw in forbidden):
        return "Only SELECT queries are permitted."
    conn = sqlite3.connect(DB_PATH)
    cur = conn.execute(query)
    rows = cur.fetchmany(20)
    headers = [d[0] for d in cur.description]
    conn.close()
    lines = [", ".join(headers)] + [", ".join(str(v) for v in row) for row in rows]
    return "\n".join(lines)


# Test the safety guard, then a real query
print(run_sql.invoke({"query": "DROP TABLE sales"}))
print()
print(run_sql.invoke({"query": "SELECT product, amount FROM sales ORDER BY amount DESC LIMIT 3"}))

### Why the safety guard matters

Without the guard, the agent could generate a `DELETE` or `DROP` query if it misunderstood the question -- or if someone crafted a prompt to trick it. Blocking write keywords at the tool level means **even a confused or manipulated agent cannot modify the data**.

In production you would also:
- Connect with a read-only database user
- Validate the query with a SQL parser rather than keyword matching
- Cap the row limit to prevent accidental full-table scans

---

## Part 3 -- Building the ReAct Agent

`create_react_agent` from LangGraph wires the LLM and tools together into the ReAct loop automatically.
You pass in the model and a list of tools -- the framework handles the rest.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-5-nano")
app = create_react_agent(llm, tools=[list_tables, describe_table, run_sql])

print("Agent ready.")

In [ ]:
def ask(question: str, show_steps: bool = True) -> str:
    """Run a question through the agent and print the tool-call trace."""
    result = app.invoke({"messages": [HumanMessage(question)]})

    if show_steps:
        print(f"Question: {question}")
        print("-" * 56)
        for msg in result["messages"][1:]:  # skip the original HumanMessage
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    args = ", ".join(f"{k}={v!r}" for k, v in tc["args"].items())
                    print(f"  [call]   {tc['name']}({args})")
            elif msg.type == "tool":
                preview = msg.content[:120].replace("\n", " / ")
                print(f"  [result] {preview}")
        print("-" * 56)

    answer = result["messages"][-1].content
    print(f"Answer: {answer}\n")
    return answer

---

## Part 4 -- Asking Questions

Watch the `[call]` and `[result]` lines -- that is the ReAct loop made visible.
The agent discovers the schema on its own, writes SQL, reads the result, and then composes its answer.

In [ ]:
# First question: the agent explores the database from scratch
ask("What tables are in the database and what columns do they have?")

In [ ]:
# The agent writes a GROUP BY + SUM query without being told how
ask("Which product had the highest total sales?")

In [ ]:
# Filter query -- the agent knows to use WHERE
ask("Show me all North region sales above $1000")

In [ ]:
# Multi-step reasoning -- needs GROUP BY across two dimensions
ask("What was the total sales amount for each region, sorted highest to lowest?")

---

## Part 5 -- Exercises

Attempt these before scrolling to the answer key at the bottom of the notebook.
Each exercise has a starter cell -- fill in the `TODO` and run it.

In [ ]:
# Exercise 1 -- Sales count per product
# Ask the agent: how many individual sales were recorded per product?
# (You want a COUNT of rows, not a SUM of amounts.)
# Expected result: Widget A = 3, Widget B = 3, Widget C = 2

# TODO: write the ask() call


In [ ]:
# Exercise 2 -- Q1 2024 sales
# Ask the agent: which sales happened in Q1 2024 (January through March)?
# Expected result: 6 rows (ids 1-6); ids 7 and 8 are in April and should be excluded

# TODO: write the ask() call


In [ ]:
# Exercise 3 -- Custom tool
# Add a new @tool called `average_by` that takes a column name and returns
# the average sale amount grouped by that column.
# Create a new agent that includes it, then ask:
# "What is the average sale amount by region?"

# TODO: define the @tool
# TODO: create new agent with the extra tool
# TODO: run the question


---
---

# Answer Key

Review these only after attempting the exercises yourself.

### Exercise 1 -- Sales count per product

The agent should call `list_tables` -> `describe_table` -> `run_sql` with a `GROUP BY product` + `COUNT(*)` query.
The key distinction: "how many" maps to `COUNT(*)`, not `SUM(amount)`.

In [ ]:
ask("How many individual sales were recorded per product?")

The agent typically generates:
```sql
SELECT product, COUNT(*) AS sales_count
FROM sales
GROUP BY product
ORDER BY sales_count DESC
```
Result: Widget A = 3, Widget B = 3, Widget C = 2.

### Exercise 2 -- Q1 2024 sales

The agent must interpret "Q1 2024" as a date range and filter with `WHERE date BETWEEN` or `WHERE date < '2024-04-01'`.

In [ ]:
ask("Which sales happened in Q1 2024 (January through March)?")

The agent typically generates:
```sql
SELECT *
FROM sales
WHERE date BETWEEN '2024-01-01' AND '2024-03-31'
```
Result: 6 rows (ids 1-6). Ids 7 and 8 are April and are excluded.

> **Why this works:** the LLM knows that Q1 means January-March because that knowledge is baked into its training. The agent translates a business concept into a SQL date predicate without any hardcoded mapping.

### Exercise 3 -- Custom `average_by` tool

A purpose-built tool gives the agent a cleaner path to common aggregations.
Notice how naming and docstrings guide which tool the agent reaches for.

In [ ]:
@tool
def average_by(column: str) -> str:
    """Return the average sale amount grouped by the specified column (e.g. region, product)."""
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(
        f"SELECT {column}, ROUND(AVG(amount), 2) AS avg_amount "
        f"FROM sales GROUP BY {column} ORDER BY avg_amount DESC"
    ).fetchall()
    conn.close()
    return "\n".join(f"{r[0]}: ${r[1]:.2f}" for r in rows)


# New agent that includes the extra tool
app_extended = create_react_agent(llm, tools=[list_tables, describe_table, run_sql, average_by])

result = app_extended.invoke(
    {"messages": [HumanMessage("What is the average sale amount by region?")]}
)
print(result["messages"][-1].content)

> **Notice:** with `average_by` available, the agent may reach for it directly rather than writing a GROUP BY query from scratch -- it is a shorter, more reliable path to the answer. This is a general principle: **well-named, focused tools shape agent behaviour** more reliably than prompt engineering alone.

---

## What's Next?

- **Example 15** -- [CrewAI Research Crew](../15-crewai-research-crew/README.md): multi-agent orchestration with a different framework
- **LangGraph docs** -- `create_react_agent` and tool calling: https://langchain-ai.github.io/langgraph/
- **Production SQL agents** -- swap SQLite for Postgres, use `sqlalchemy` with parameterised queries, connect as a read-only user

### Key takeaways

| Concept | What to remember |
|---------|------------------|
| ReAct loop | Reason -> Act -> Observe, repeated until done |
| `create_react_agent` | Wires LLM + tools into the loop; you just supply the pieces |
| Tool docstrings | The LLM reads them -- write clear, specific descriptions |
| Safety guard | Block writes at the tool level, not in the prompt |
| Schema discovery | The agent finds columns itself -- no hardcoded SQL needed |